[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_SpeechRecognition.ipynb)

# Benchmark: Speech Recognition

Scores a pretrained ASR model's Word Error Rate against gold-labeled data, using
`sparknlp.benchmark.Benchmark.evaluate(..., task="speechrecognition")`.

**Dataset**: [`hf-internal-testing/librispeech_asr_dummy`](https://huggingface.co/datasets/hf-internal-testing/librispeech_asr_dummy),
a small slice of LibriSpeech `dev-clean` commonly used for smoke-testing ASR pipelines.

**Model**: `Wav2Vec2ForCTC.pretrained()` (default: `asr_wav2vec2_base_960h`).

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-09-18 02:07:52--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-09-18 02:07:53--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’

-                   100%[===================>]   1.45K  --.-KB/s    in 0s      

2026-09-18 02:07:53 (32.5 MB/s) - written to stdout [1483/1483]

Installing PySpa

In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jre-headless_17.0.20+8-1~24.04_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.20+8-1~24.04) ...
Selecting previously unselected package openjdk-17-jdk-headless:amd64.
Preparing to unpack .../openjdk-17-jdk-headless_17.0.20+8-1~24.04_amd64.deb ...
Unpacking openjdk-17-jdk-headless:amd64 (17.0.20+8-1~24.04) ...
Setting up openjdk-17-jre-headless:amd64 (17.0.20+8-1~24.04) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.028s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.028s][warning]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
!pip install -q soundfile
from sparknlp.base import AudioAssembler
from sparknlp.annotator import Wav2Vec2ForCTC
from pyspark.ml import Pipeline
from pyspark.sql.types import StructType, StructField, ArrayType, FloatType, LongType, StringType
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

In [8]:
import io
import pandas as pd
import soundfile as sf

url = ("https://huggingface.co/api/datasets/hf-internal-testing/librispeech_asr_dummy"
       "/parquet/clean/validation/0.parquet")
df = pd.read_parquet(url).head(15)

rows = []
for _, row in df.iterrows():
    data, sample_rate = sf.read(io.BytesIO(row["audio"]["bytes"]))
    rows.append(([float(x) for x in data], int(sample_rate), row["text"]))

print(len(rows), "audio samples")
print(rows[0][2])

15 audio samples
MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL

> **Note: WER is case-sensitive by default, on both the Scala and Python sides of
> `Benchmark` -- matching how [jiwer](https://github.com/jitsi/jiwer) itself scores by default.**
> LibriSpeech's own reference transcripts are natively upper-case, which happens to match
> `Wav2Vec2ForCTC`'s own upper-case output convention here, so we leave both alone. If your
> model or your gold data use a different case convention, normalize them to match *before*
> calling `Benchmark.evaluate` -- otherwise every word looks like a mismatch even when the
> transcription is right.

In [10]:
schema = StructType([
    StructField("audio_content", ArrayType(FloatType())),
    StructField("sampling_rate", LongType()),
    StructField("label", StringType()),
])
gold_data = spark.createDataFrame(rows, schema)
gold_data.select("sampling_rate", "label").show(5, truncate=60)

+-------------+------------------------------------------------------------+
|sampling_rate|                                                       label|
+-------------+------------------------------------------------------------+
|        16000|MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND W...|
|        16000|NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS ...|
|        16000|HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH ...|
|        16000|HE HAS GRAVE DOUBTS WHETHER SIR FREDERICK LEIGHTON'S WORK...|
|        16000|LINNELL'S PICTURES ARE A SORT OF UP GUARDS AND AT EM PAIN...|
+-------------+------------------------------------------------------------+
only showing top 5 rows

## 2. Build the pipeline

In [12]:
audio_assembler = AudioAssembler().setInputCol("audio_content").setOutputCol("audio_assembler")
speech_to_text = Wav2Vec2ForCTC.pretrained() \
    .setInputCols(["audio_assembler"]).setOutputCol("text")

pipeline = Pipeline(stages=[audio_assembler, speech_to_text])
pipeline_model = pipeline.fit(gold_data)

asr_wav2vec2_base_960h download started this may take some time.
Approximate size to download 217 MB
[OK!]

## 3. Run the benchmark

In [14]:
report = Benchmark.evaluate(
    pipeline_model, gold_data, task="speechrecognition", text_col="audio_content", label_col="label")
print(report)

speechrecognition accuracy (n=15, scored: text): wer=0.1522

## Reading the result

A nonzero WER here is expected -- the CTC decoder's own minor misspellings (e.g. missed
apostrophes, doubled letters) count as word errors even when the transcription is clearly
correct to a human reader.